# Build email extractor

In [1]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from pydantic_ai import Agent
from constants import MODEL_LARGE
from dotenv import load_dotenv 

load_dotenv()

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="Summarize the email in 3-4 sentences.")

email_extractor_agent = Agent(MODEL_LARGE, system_prompt="""
    You are customer support agent, your task is to 
    extract relevant information from an email.
""", output_type=EmailExtractor)



TODO: load in the data and test the agent on one email

In [6]:
import pandas as pd 

df = pd.read_json("emails_cleaned.json")
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


In [7]:
sample_mail = df.iloc[2]["inputs"]["email"]
sample_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [4]:
result = await email_extractor_agent.run(sample_mail)
result

AgentRunResult(output=EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson reports being locked out of his account for three days due to invalid credentials despite entering the correct password. He has attempted password reset multiple times, but the reset emails never arrive, even after checking spam/junk and trying different browsers and devices. He needs urgent access to important work documents before a Friday deadline and requests assistance to verify his identity and restore account access.'))

In [6]:
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson reports being locked out of his account for three days due to invalid credentials despite entering the correct password. He has attempted password reset multiple times, but the reset emails never arrive, even after checking spam/junk and trying different browsers and devices. He needs urgent access to important work documents before a Friday deadline and requests assistance to verify his identity and restore account access.')

In [7]:
isinstance(result.output, BaseModel)

True

In [8]:
result.output.summary

'Oscar Johansson reports being locked out of his account for three days due to invalid credentials despite entering the correct password. He has attempted password reset multiple times, but the reset emails never arrive, even after checking spam/junk and trying different browsers and devices. He needs urgent access to important work documents before a Friday deadline and requests assistance to verify his identity and restore account access.'

TODO: show ground truth and compare

## Load in prompts from mlflow

In [5]:
from mlflow.genai import load_prompt

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(
        description=load_prompt("email-urgency-description").format()
    )
    summary: str = Field(
        description=load_prompt("summary-description").format(num_sentences=4)
    )


email_extractor_agent = Agent(
    MODEL_LARGE,
    system_prompt=load_prompt("email-extractor-system-prompt").format(),
    output_type=EmailExtractor,
)

In [9]:
result = await email_extractor_agent.run(sample_mail)
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson has been locked out of his account for three days due to invalid credentials despite entering the correct password. He has attempted password reset multiple times but reset emails never arrive, even after checking spam/junk folders and trying different browsers and devices. This prevents him from accessing important work documents needed to meet an upcoming Friday deadline. He requests urgent assistance and offers to verify his identity.')

## LLM Judge 

1. require data with columns: inputs, expectations, outputs
2. mlflow experiments

In [11]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_category': 'technical',
 'urgency': 'high',
 'summary': 'Oscar Johansson has been locked out of his account for three days due to invalid credentials despite entering the correct password. He has attempted password reset multiple times but reset emails never arrive, even after checking spam/junk folders and trying different browsers and devices. This prevents him from accessing important work documents needed to meet an upcoming Friday deadline. He requests urgent assistance and offers to verify his identity.'}

TODO: dataframe with inputs, expectations, outputs but only 1 row

In [22]:
df["outputs"] = [{},{},result.output.model_dump(), {}]
df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea...",{}


In [23]:
df_sample = df.drop([0,1,3])
df_sample

,inputs,expectations,outputs
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."


## LLM judge

In [25]:
from mlflow.genai.scorers import get_all_scorers

# get_all_scorers()

In [26]:
from mlflow.genai.scorers import Correctness, Summarization, Completeness, Fluency
import mlflow

llm_judge = "openrouter:/nvidia/nemotron-3-nano-30b-a3b:free"

with mlflow.start_run(run_name="email-extractor-evaluation"):
    mlflow.log_param("model_judge", llm_judge)
    mlflow.log_param("model", MODEL_LARGE)

    results = mlflow.genai.evaluate(
        data = df_sample,
        scorers=[
            Correctness(model=llm_judge),
            Summarization(model=llm_judge)
        ]
    )

results

2026/04/15 14:53:52 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
Evaluating: 100%|██████████| 1/1 [Elapsed: 00:06, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: email-extractor-evaluation
  Run ID: 80a0132a96e948babfa8656c73cc10e1

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



EvaluationResult(
  run_id: 80a0132a96e948babfa8656c73cc10e1
  metrics:
    summarization/mean: 1.0
    correctness/mean: 1.0
  result_df: 1 rows x 15 cols
)

In [28]:
results.result_df

,trace_id,summarization/value,correctness/value,expected_response/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-c440ef348be85fea8ef589efb6b2f823,yes,yes,"{""sender_name"": ""Oscar Johansson"", ""sender_ema...","{""info"": {""trace_id"": ""tr-c440ef348be85fea8ef5...",None,OK,1776257633507,0,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema...","{'mlflow.user': 'aigineer', 'mlflow.source.nam...",{'mlflow.artifactLocation': '/Users/aigineer/D...,"[{'trace_id': 'xEDvNIvoX+qO9YnvtrL4Iw==', 'spa...",[{'assessment_id': 'a-e53379f39eab4b728b917902...
